# 1. Preprocessing and artifact labels

Batch 1 of the artifact-cleaning project (see `README.md` in this folder).
This notebook:

1. preprocesses each subject's recording and **automatically labels**
   blinks, saccades and (weak) EMG artifacts, then shows how often each
   occurs per condition;
2. writes **review sheets** for hand spot-checking a subset of the labels;
3. reports the labels' **precision and miss rate** from the filled-in
   sheets.

All the work happens in the modules under `preprocessing/` and `labels/`;
the cells here only choose settings, run steps and show results.
Preprocessing is cached per subject, so re-running is fast after the
first time. Everything is written to
`<dataset>/cocktail/bids/derivatives/artifacts/`.

In [ ]:
import sys
from pathlib import Path

# This notebook runs from inside artifacts/; the repo root (one level up)
# has to be on the path for `import artifacts` to work.
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns

from artifacts import config
from artifacts.labels import pipeline
from artifacts.labels.review import plot_event, segment_of
from artifacts.labels.store import load_labels
from artifacts.preprocessing.loading import resolve_subjects
from artifacts.preprocessing.preprocess import load_preprocessed

print('BIDS dataset:', config.BIDS_ROOT)
print('outputs:     ', config.DERIV_ROOT)

## Settings

In [ ]:
SUBJECTS = resolve_subjects('all')  # or a list, e.g. ['01', '02']
REBUILD = False                     # True: redo preprocessing even if cached

REVIEW_SUBJECTS = SUBJECTS[:3]      # subjects to spot-check by hand (step 2)
N_PER_TYPE = 20                     # labelled events per type, per subject
N_UNLABELLED = 20                   # random unlabelled windows, per subject (for the miss rate)

CONDITION_ORDER = list(config.CONDITIONS)
SUBJECTS

## 1. Preprocess and auto-label

The first run per subject reads its ~1 h recording one speech segment at a
time, so it takes a while; after that the preprocessed data is cached.

The table below records the preprocessing choices made per subject.
Worth checking once:
- `line_freq_source` should be `detected` (a clear 50/60 Hz peak was
  found). `bids` or `none` means no clear mains peak, so the notch used
  the dataset's metadata, or wasn't applied.
- `eog_bads` should be empty: ocular labels from a bad EOG electrode
  are unreliable.
- `interpolated` lists the bad channels (from `analysis/experiment.py`'s
  `mark_bad_channels()`) that were rebuilt from their neighbours.

In [ ]:
summary, preprocessing = pipeline.label_subjects(SUBJECTS, rebuild=REBUILD)
preprocessing

Events per minute and % of time labelled, per subject and condition
(also saved as `group/label_summary.tsv`):

In [ ]:
summary

### Event rates and labelled time, by condition

Bars are the mean over subjects (error bars: SEM), dots are individual
subjects. Typical blink rates while listening are roughly 10-20 per
minute; a subject far outside that, or an EMG rate far above the rest,
is worth a look in the review sheets.

In [ ]:
rate_cols = [c for c in summary if c.endswith('_per_min')]
pct_cols = [c for c in summary if c.endswith('_pct_time')]
rates = summary.melt(['subject', 'condition'], rate_cols, var_name='type', value_name='per minute')
rates['type'] = rates['type'].str.removesuffix('_per_min')
pct = summary.melt(['subject', 'condition'], pct_cols, var_name='type', value_name='% of time')
pct['type'] = pct['type'].str.removesuffix('_pct_time')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, data, y in ((axes[0], rates, 'per minute'), (axes[1], pct, '% of time')):
    sns.barplot(data, x='type', y=y, hue='condition', hue_order=CONDITION_ORDER, errorbar='se', ax=ax, alpha=0.6)
    sns.stripplot(data, x='type', y=y, hue='condition', hue_order=CONDITION_ORDER, dodge=True,
                  ax=ax, size=4, legend=False)
    ax.set_xlabel('')
axes[0].set_title('events per minute')
axes[1].set_title('% of time labelled ("any" = labelled as anything)')
axes[1].get_legend().remove()
fig.tight_layout()

### Example events

One randomly chosen auto-labelled event of each type from the first
subject, drawn the way the review sheets show them: EOG channels in red,
the labelled window shaded.

In [ ]:
example_subject = SUBJECTS[0]
raw, segments, _ = load_preprocessed(example_subject)
for kind in ('ocular', 'emg'):
    events = load_labels(example_subject, kind, reviewed=False)
    for trial_type, group in events.groupby('trial_type'):
        row = group.sample(1, random_state=0).iloc[0].to_dict()
        row['id'] = f'{example_subject}-{trial_type} (example)'
        plot_event(raw, row, segment_of(row['onset'], segments))
        plt.show()

## 2. Review sheets

For each review subject and label kind (`ocular`, `emg`), this draws
`N_PER_TYPE` random labelled events per type plus `N_UNLABELLED` random
*unlabelled* 1 s windows, and writes one PNG each plus a `review.csv`
under `derivatives/artifacts/review/<kind>/sub-XX/`.

Open the PNGs and fill in the `verdict` column of each `review.csv`:
- labelled events: `ok`, `wrong` or `unsure` (optionally
  `corrected_onset` / `corrected_duration`, in seconds)
- unlabelled windows (`trial_type` = `none`): `clean`, or
  `missed:<type>` (e.g. `missed:blink`) if an artifact was there

A sheet that already has verdicts is never overwritten (pass
`overwrite=True` to replace it). Reviewed verdicts are applied whenever
labels are loaded later: events marked `wrong` are dropped and
corrections applied. **EMG labels must be reviewed before batch 3 trains
on them.**

In [ ]:
sheets = pipeline.write_review_sheets(REVIEW_SUBJECTS, n_per_type=N_PER_TYPE, n_unlabelled=N_UNLABELLED)
sheets

## 3. Label accuracy from the reviews

Run this after filling in the sheets. **Precision** (per type): of the
labelled events reviewed, the fraction that were real. **Miss rate**: of
the random unlabelled windows, the fraction that hid an artifact. These
are the numbers to report for the auto-labeller. Also saved as
`group/review_summary_<kind>.tsv`.

In [ ]:
reviews = pipeline.review_summaries()
if not reviews:
    print('No filled-in review sheets yet.')
for kind, table in reviews.items():
    print(kind)
    display(table)

In [ ]:
if reviews:
    fig, axes = plt.subplots(1, len(reviews), figsize=(5 * len(reviews), 3.5), squeeze=False)
    for ax, (kind, table) in zip(axes[0], reviews.items()):
        labelled = table[table['trial_type'] != 'none']
        ax.bar(labelled['trial_type'], labelled['precision'], color='0.5')
        miss = table.loc[table['trial_type'] == 'none', 'miss_rate']
        ax.set_title(kind + (f'  (miss rate {miss.iloc[0]:.0%})' if len(miss) else ''))
        ax.set_ylim(0, 1)
        ax.set_ylabel('precision')
    fig.tight_layout()